# sre-gym — GRPO on-policy training against the live env

Open in Colab:

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dakshdoesdev/sre-enginnerllm/blob/main/train/grpo_run.ipynb)

**Run AFTER `train/sanity_run.ipynb`.** Loads the SFT checkpoint from `dakshdoesdev/sre-gym-qwen35-4b-sft`, runs ~300 GRPO steps where each step samples a scenario, rolls out K=4 trajectories with the current policy, and updates weights on group-relative advantages.

**Reward = env.evaluate()['score']** — deterministic scalar from the grader. No PRM, no LLM-as-judge.

**Before running**:
- Same secrets as `sanity_run.ipynb` (`HFTOKEN`, optional `WANDB_API_KEY`)
- Runtime → A100 GPU
- The notebook auto-clones the repo and boots the env's pool server in-Colab

**Architecture**

```
Colab A100 40GB
├── uvicorn: pool_server on :8100  (env instances + lease TTL)
└── this notebook:
    ├── FastLanguageModel.from_pretrained(SFT_ADAPTER_REPO, load_in_4bit=True)
    ├── GRPO rollout loop:
    │     for step in range(NUM_STEPS):
    │         scenario = sample_scenario()
    │         for k in range(K):
    │             rollout = play_episode(policy, pool_server, scenario)
    │             advantage[k] = rollout.final_score - group_mean
    │         policy_gradient_update(model, rollouts, advantages)
    └── push_to_hub('dakshdoesdev/sre-gym-qwen35-4b-grpo') every 25 steps
```

Expected wall-clock: ~4–6h. Expected GPU memory: ~24 GB.

## 0. Runtime + deps

In [ ]:
!nvidia-smi
!python -c 'import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available())'

In [ ]:
%%bash
pip install -q --upgrade pip
pip install -q "unsloth[colab-new]>=2025.12,<2026.06"
pip install -q "unsloth_zoo>=2025.12,<2026.06"
pip install -q "trl>=0.12,<0.16" "transformers>=4.48,<4.60" "peft>=0.14,<0.20" "accelerate>=1.2,<2.0"
pip install -q "datasets>=3.0,<4.0" "wandb>=0.18,<1.0" "bitsandbytes>=0.45" httpx fastapi uvicorn pydantic

## 1. Clone repo + boot the pool server

The pool server exposes our env as `/allocate /reset /exec_tool /evaluate /close`. We run it in the same Colab VM, in a background thread, so the GRPO rollout loop hits `http://127.0.0.1:8100`.

In [ ]:
import subprocess, os, time

if not os.path.exists('/content/sre-enginnerllm'):
    subprocess.check_call(['git', 'clone', 'https://github.com/dakshdoesdev/sre-enginnerllm.git', '/content/sre-enginnerllm'])
os.chdir('/content/sre-enginnerllm')
subprocess.check_call(['pip', 'install', '-q', '-e', '.'])

# Boot the pool server in the background
pool_proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'openclaw_integration.pool_server:app',
     '--host', '127.0.0.1', '--port', '8100'],
    stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
)
time.sleep(5)
import httpx
print(httpx.get('http://127.0.0.1:8100/healthz', timeout=10).json())

## 2. Config

In [ ]:
import os

# --- Colab secrets bridge (HFTOKEN, WANDB) ---
try:
    from google.colab import userdata  # type: ignore
    for _src in ("HF_TOKEN", "HFTOKEN", "HUGGINGFACE_TOKEN"):
        try:
            v = userdata.get(_src)
            if v:
                os.environ["HF_TOKEN"] = v
                break
        except Exception:
            pass
    for _src in ("WANDB_API_KEY", "WANDB", "wandb"):
        try:
            v = userdata.get(_src)
            if v:
                os.environ["WANDB_API_KEY"] = v
                break
        except Exception:
            pass
except ImportError:
    pass

if "HF_TOKEN" not in os.environ:
    raise RuntimeError(
        "HF_TOKEN not set. Add `HFTOKEN` (or `HF_TOKEN`) in Colab Secrets "
        "(left sidebar 🔑 panel) and toggle 'Notebook access' on."
    )
# ---

SFT_ADAPTER_REPO = 'dakshdoesdev/sre-gym-qwen35-4b-sft'   # output of sanity_run.ipynb
GRPO_ADAPTER_REPO = 'dakshdoesdev/sre-gym-qwen35-4b-grpo' # destination

NUM_GRPO_STEPS = 300        # total policy updates
ROLLOUTS_PER_STEP = 4       # K (group size)
MAX_TICKS = 12              # per-rollout cap; matches env's scenario.max_ticks
LEARNING_RATE = 5e-6        # much lower than SFT — GRPO needs gentle updates
KL_COEFF = 0.05             # reserved for v2 (PPO-clip + KL)
PPO_CLIP = 0.2
SAVE_EVERY = 25             # push adapter to Hub every N steps
SNAPSHOT_STEPS = [0, 50, 100, 150, 200, 250]  # save full rollout JSONL here
POOL_URL = 'http://127.0.0.1:8100'

# Drive checkpoint path — survives Colab disconnect even if HF push fails.
# Mount Drive on first use; we'll skip silently if Drive can't mount.
DRIVE_CKPT_DIR = '/content/drive/MyDrive/sre-gym-grpo'
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)
    os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
    DRIVE_OK = True
except Exception as exc:
    print(f"[drive] mount failed ({exc}); will only push to HF Hub")
    DRIVE_OK = False

# Baseline reference values for plotting (from train/data/eval_sweep_baselines.jsonl
# and the README hero table). wandb captures these so the training curve auto-renders
# with horizontal reference lines.
BASELINES = {
    'random':              0.345,
    'heuristic':           0.187,
    'llama70b_groq':       0.421,
    'llama70b_fireworks':  0.725,
    'scripted_optimal':    0.760,
    'claude_opus':         0.769,
}

os.environ.setdefault('WANDB_PROJECT', 'sre-gym-grpo')
os.environ.setdefault('WANDB_RUN_NAME', 'qwen35-4b-grpo-300')
os.environ.setdefault(
    'WANDB_MODE',
    'online' if os.environ.get('WANDB_API_KEY') else 'offline',
)

print(f"HF_TOKEN set:      {'yes' if os.environ.get('HF_TOKEN') else 'NO'}")
print(f"WANDB_API_KEY set: {'yes (online)' if os.environ.get('WANDB_API_KEY') else 'no (offline mode)'}")
print(f"Drive ckpt:        {'yes -> ' + DRIVE_CKPT_DIR if DRIVE_OK else 'no (HF Hub only)'}")
print(f"SFT source:        {SFT_ADAPTER_REPO}")
print(f"GRPO destination:  {GRPO_ADAPTER_REPO}")
print(f"Snapshot steps:    {SNAPSHOT_STEPS}")

## 3. Load SFT checkpoint + LoRA for continued training

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_ADAPTER_REPO,
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True,
)
# SFT adapter is already attached; just enable training mode.
model = FastLanguageModel.get_peft_model(
    model,
    r=32, lora_alpha=32, lora_dropout=0.0, bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth', random_state=42,
)

## 4. Rollout helper — play one episode with the current policy

In [ ]:
import json, httpx, uuid

SYSTEM_PROMPT = (
    'You are an SRE agent operating inside the sre-gym environment. '
    'On each turn, emit exactly one UnifiedIncidentAction JSON object — no surrounding prose, '
    'no code fences, no commentary. Valid action_types: query_logs, query_metrics, '
    'query_dependencies, query_deploys, rollback_deploy, restart_service, isolate_service, '
    'run_check, submit_hypothesis, escalate, declare_resolved.'
)

def generate_action(prompt_text, max_new_tokens=96):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': prompt_text},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to('cuda')
    with torch.no_grad():
        out = model.generate(
            input_ids=inputs, max_new_tokens=max_new_tokens,
            temperature=0.7, do_sample=True, top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True).strip()
    return text, inputs, out

def rollout_episode(scenario_id):
    """Play one full episode. Returns (trajectory, final_score)."""
    with httpx.Client(base_url=POOL_URL, timeout=60) as c:
        lease = c.post('/allocate', json={'task_key': scenario_id}).json()['lease_id']
        reset = c.post('/reset', json={'lease_id': lease, 'scenario_id': scenario_id}).json()
        obs = json.loads(reset['observation'])
        traj = []
        for tick in range(MAX_TICKS):
            if obs.get('done'):
                break
            prompt_text = obs['prompt_text']
            gen_text, inp_ids, out_ids = generate_action(prompt_text)
            # Parse action — on parse failure, let env return wrong_remediation_target
            try:
                action = json.loads(gen_text)
                tool_name = action.pop('action_type')
                args = action
            except Exception:
                tool_name, args = 'escalate', {}
            step_resp = c.post('/exec_tool', json={
                'lease_id': lease,
                'tool_call': {'name': tool_name, 'arguments': args},
            }).json()
            next_obs = json.loads(step_resp['observation'])
            traj.append({
                'prompt': prompt_text,
                'response': gen_text,
                'input_ids': inp_ids.cpu(),
                'output_ids': out_ids.cpu(),
                'reward': float(next_obs.get('reward') or 0.0),
            })
            obs = next_obs
        ev = c.post('/evaluate', json={'lease_id': lease}).json()
        c.post('/close', json={'lease_id': lease})
        return traj, float(ev['score'])

## 5. GRPO loss + update step

In [ ]:
import torch
import torch.nn.functional as F
from torch.optim import AdamW

optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=LEARNING_RATE)
FastLanguageModel.for_training(model)

def per_token_logprobs(input_ids, output_ids):
    """Compute per-response-token logprobs for a (prompt, full_output) pair."""
    # full_seq = prompt tokens + response tokens
    full_seq = output_ids.to('cuda')
    prompt_len = input_ids.shape[-1]
    # Forward pass — we need logits for positions [prompt_len-1 .. end-1]
    logits = model(full_seq).logits[0]
    response_logits = logits[prompt_len - 1 : -1]  # [resp_len, vocab]
    response_tokens = full_seq[0, prompt_len:]      # [resp_len]
    logprobs = F.log_softmax(response_logits, dim=-1)
    token_logprobs = logprobs.gather(-1, response_tokens.unsqueeze(-1)).squeeze(-1)
    return token_logprobs

def grpo_step(trajectories, advantages):
    """Apply GRPO update to a group of trajectories.

    trajectories: list of list of turn dicts (one outer list per rollout)
    advantages: list[float] — one advantage per rollout, broadcast to all turns
    """
    optimizer.zero_grad()
    total_loss = 0.0
    n_turns = 0
    for traj, adv in zip(trajectories, advantages):
        for turn in traj:
            if turn['input_ids'] is None:
                continue
            token_lp = per_token_logprobs(turn['input_ids'], turn['output_ids'])
            # Simple policy gradient: -adv * mean_logprob (no PPO-clip baseline yet)
            turn_loss = -adv * token_lp.mean()
            total_loss = total_loss + turn_loss
            n_turns += 1
    if n_turns == 0:
        return 0.0
    loss = total_loss / n_turns
    loss.backward()
    torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
    optimizer.step()
    return float(loss.detach().cpu())

## 6. Main GRPO loop

In [ ]:
import random, statistics, shutil, time
from collections import defaultdict, Counter
from pathlib import Path
import wandb

# Fetch scenarios from the pool server
scenarios = httpx.get(f'{POOL_URL}/healthz').json()['scenarios']
# Hold out 3 scenarios for eval (one per difficulty): drop procgen variants of these
HELD_OUT = {'worker_deploy_cascade__p04', 'db_config_rollout__p04', 'gateway_auth_rollout__p04'}
train_scenarios = [s for s in scenarios if s not in HELD_OUT]
print(f'training on {len(train_scenarios)} scenarios, holding out {len(HELD_OUT)}')

run = wandb.init(
    project=os.environ.get('WANDB_PROJECT', 'sre-gym-grpo'),
    name=os.environ.get('WANDB_RUN_NAME', 'qwen35-4b-grpo-300'),
    config={
        'sft_adapter': SFT_ADAPTER_REPO,
        'grpo_destination': GRPO_ADAPTER_REPO,
        'num_steps': NUM_GRPO_STEPS,
        'rollouts_per_step': ROLLOUTS_PER_STEP,
        'learning_rate': LEARNING_RATE,
        'baselines': BASELINES,
    },
)

# Trajectory snapshot output (also pushed to Hub at end alongside the model)
SNAPSHOT_DIR = Path('/content/grpo_snapshots')
SNAPSHOT_DIR.mkdir(exist_ok=True)
TRAINING_LOG = Path('/content/grpo_training.log')


def _strip_torch_from_traj(traj):
    """Remove torch tensors before JSON-serializing a trajectory."""
    return [
        {
            'prompt': t['prompt'],
            'response': t['response'],
            'reward': t['reward'],
        }
        for t in traj
    ]


def save_snapshot(step, scenario, rollouts, scores):
    """Persist one snapshot per checkpoint step: best, worst, and median rollouts."""
    sorted_pairs = sorted(zip(scores, rollouts), key=lambda p: p[0])
    snapshot = {
        'step': step,
        'scenario_id': scenario,
        'scores': scores,
        'best_rollout': {
            'final_score': sorted_pairs[-1][0],
            'trajectory': _strip_torch_from_traj(sorted_pairs[-1][1]),
        },
        'worst_rollout': {
            'final_score': sorted_pairs[0][0],
            'trajectory': _strip_torch_from_traj(sorted_pairs[0][1]),
        },
    }
    out = SNAPSHOT_DIR / f'snapshot_step{step:04d}.jsonl'
    with out.open('w') as f:
        f.write(json.dumps(snapshot) + '\n')
    return out


def append_log(line):
    with TRAINING_LOG.open('a') as f:
        f.write(line + '\n')


running_reward = 0.0
running_solved = 0.0
scenario_window = defaultdict(list)  # scenario_id -> recent scores (last 20)
action_counter = Counter()  # parsed action_type frequency over all rollouts
training_started = time.time()

for step in range(NUM_GRPO_STEPS):
    scenario = random.choice(train_scenarios)
    FastLanguageModel.for_inference(model)
    rollouts = []
    scores = []
    for k in range(ROLLOUTS_PER_STEP):
        traj, score = rollout_episode(scenario)
        rollouts.append(traj)
        scores.append(score)
        # Track action distribution
        for turn in traj:
            try:
                act = json.loads(turn['response']).get('action_type', 'parse_error')
            except Exception:
                act = 'parse_error'
            action_counter[act] += 1

    mean_score = statistics.mean(scores)
    score_std = statistics.stdev(scores) if len(scores) > 1 else 0.0
    advantages = [s - mean_score for s in scores]
    resolved_rate = sum(1 for s in scores if s > 0.6) / len(scores)

    FastLanguageModel.for_training(model)
    loss = grpo_step(rollouts, advantages)

    running_reward = 0.9 * running_reward + 0.1 * mean_score if step > 0 else mean_score
    running_solved = 0.9 * running_solved + 0.1 * resolved_rate if step > 0 else resolved_rate
    scenario_window[scenario].append(mean_score)
    if len(scenario_window[scenario]) > 20:
        scenario_window[scenario].pop(0)

    elapsed_h = (time.time() - training_started) / 3600.0

    log_payload = {
        'step': step,
        'loss': loss,
        'mean_score': mean_score,
        'score_std': score_std,
        'running_reward': running_reward,
        'running_solved_rate': running_solved,
        'min_score': min(scores),
        'max_score': max(scores),
        'adv_spread': max(advantages) - min(advantages),
        'scenario': scenario,
        'elapsed_hours': elapsed_h,
        # baselines as constant lines for the chart
        **{f'baseline/{k}': v for k, v in BASELINES.items()},
        # per-scenario rolling means
        **{f'per_scenario_mean/{s}': sum(window) / len(window)
           for s, window in scenario_window.items() if window},
    }
    wandb.log(log_payload)

    line = (f'[{step:03d} | {elapsed_h:5.2f}h] {scenario:<35} '
            f'scores={[f"{s:.2f}" for s in scores]} mean={mean_score:.3f} '
            f'std={score_std:.3f} loss={loss:.4f} run_reward={running_reward:.3f}')
    print(line)
    append_log(line)

    # Save trajectory snapshot at scheduled steps
    if step in SNAPSHOT_STEPS:
        path = save_snapshot(step, scenario, rollouts, scores)
        print(f'  [snapshot] -> {path}')

    # Checkpoint: HF Hub + Drive (defense in depth)
    if (step + 1) % SAVE_EVERY == 0:
        model.save_pretrained('/content/grpo_ckpt')
        tokenizer.save_pretrained('/content/grpo_ckpt')
        try:
            model.push_to_hub(GRPO_ADAPTER_REPO)
            print(f'  [hub] pushed adapter at step {step+1} -> {GRPO_ADAPTER_REPO}')
        except Exception as exc:
            print(f'  [hub] push failed ({exc})')
        if DRIVE_OK:
            try:
                drive_target = Path(DRIVE_CKPT_DIR) / f'step{step+1:04d}'
                shutil.copytree('/content/grpo_ckpt', drive_target, dirs_exist_ok=True)
                print(f'  [drive] saved adapter -> {drive_target}')
            except Exception as exc:
                print(f'  [drive] save failed ({exc})')

# Action distribution for the whole run
print('\naction distribution over training:')
for act, n in action_counter.most_common():
    print(f'  {act:<22} {n}')
wandb.log({f'action_dist/{a}': n for a, n in action_counter.items()})

## 7. Final push + held-out eval

In [ ]:
import shutil
from huggingface_hub import HfApi

# Final adapter push (HF Hub + Drive)
model.save_pretrained('/content/grpo_ckpt')
tokenizer.save_pretrained('/content/grpo_ckpt')
try:
    model.push_to_hub(GRPO_ADAPTER_REPO)
    print(f'final adapter pushed -> {GRPO_ADAPTER_REPO}')
except Exception as exc:
    print(f'final hub push failed ({exc})')

if DRIVE_OK:
    drive_target = Path(DRIVE_CKPT_DIR) / 'final'
    shutil.copytree('/content/grpo_ckpt', drive_target, dirs_exist_ok=True)
    print(f'final adapter saved -> {drive_target}')

# Push trajectory snapshots + training log into the model repo for provenance
api = HfApi()
try:
    for f in SNAPSHOT_DIR.glob('snapshot_step*.jsonl'):
        api.upload_file(
            path_or_fileobj=str(f),
            path_in_repo=f'training_artifacts/{f.name}',
            repo_id=GRPO_ADAPTER_REPO,
            repo_type='model',
        )
    if TRAINING_LOG.exists():
        api.upload_file(
            path_or_fileobj=str(TRAINING_LOG),
            path_in_repo='training_artifacts/training.log',
            repo_id=GRPO_ADAPTER_REPO,
            repo_type='model',
        )
    print(f'pushed snapshots + log to {GRPO_ADAPTER_REPO}/training_artifacts/')
except Exception as exc:
    print(f'snapshot upload failed ({exc})')

# ---- Held-out eval comparison table ----
FastLanguageModel.for_inference(model)

EVAL_EPS = 5  # episodes per scenario per policy

import statistics
def eval_trained(scenario, n=EVAL_EPS):
    rs = [rollout_episode(scenario)[1] for _ in range(n)]
    return rs

def eval_random(scenario, n=EVAL_EPS):
    """Random valid action each turn — uses the env via pool server."""
    import random as _rnd
    rs = []
    for _ in range(n):
        with httpx.Client(base_url=POOL_URL, timeout=60) as c:
            lease = c.post('/allocate', json={'task_key': scenario}).json()['lease_id']
            obs = json.loads(c.post('/reset', json={'lease_id': lease, 'scenario_id': scenario}).json()['observation'])
            for _ in range(MAX_TICKS):
                if obs.get('done'):
                    break
                allowed = obs.get('allowed_actions') or ['query_logs']
                act_type = _rnd.choice(allowed)
                services = list(obs.get('service_health', {}).keys()) or ['database']
                args = {}
                if act_type in {'query_logs', 'query_dependencies', 'query_deploys',
                                'rollback_deploy', 'restart_service', 'isolate_service'}:
                    args = {'service': _rnd.choice(services)}
                elif act_type == 'query_metrics':
                    args = {'service': _rnd.choice(services), 'metric': 'cpu'}
                elif act_type == 'run_check':
                    args = {'check_name': 'end_to_end'}
                elif act_type == 'submit_hypothesis':
                    args = {'hypothesis': {'root_cause': 'bad_worker_deploy',
                                            'affected_services': services[:1] or ['database'],
                                            'confidence': 0.5,
                                            'recommended_next_action': 'query_logs'}}
                resp = c.post('/exec_tool', json={'lease_id': lease,
                                                   'tool_call': {'name': act_type, 'arguments': args}}).json()
                obs = json.loads(resp['observation'])
            ev = c.post('/evaluate', json={'lease_id': lease}).json()
            c.post('/close', json={'lease_id': lease})
            rs.append(float(ev['score']))
    return rs


comparison = {}
for policy_name, fn in (('random', eval_random), ('trained_3b_grpo', eval_trained)):
    rows = []
    for s in HELD_OUT:
        scores = fn(s)
        rows.append({'scenario': s, 'mean': statistics.mean(scores), 'scores': scores})
    overall = statistics.mean([r['mean'] for r in rows])
    comparison[policy_name] = {'overall_mean': overall, 'per_scenario': rows}
    print(f'\n=== {policy_name} ===  overall mean: {overall:.3f}')
    for r in rows:
        print(f"  {r['scenario']:<40} mean={r['mean']:.3f} scores={[f'{s:.2f}' for s in r['scores']]}")

# Persist comparison table for the README + pitch
out_path = '/content/grpo_eval_comparison.json'
with open(out_path, 'w') as f:
    json.dump({'baselines_from_main_repo': BASELINES, 'live_eval': comparison}, f, indent=2)
print(f'\ncomparison table -> {out_path}')

try:
    api.upload_file(
        path_or_fileobj=out_path,
        path_in_repo='training_artifacts/eval_comparison.json',
        repo_id=GRPO_ADAPTER_REPO,
        repo_type='model',
    )
    print(f'eval comparison uploaded -> {GRPO_ADAPTER_REPO}/training_artifacts/eval_comparison.json')
except Exception as exc:
    print(f'eval comparison upload failed ({exc})')

wandb.log({
    'final/eval_mean_trained_3b': comparison['trained_3b_grpo']['overall_mean'],
    'final/eval_mean_random':     comparison['random']['overall_mean'],
})
wandb.finish()
print(f'\nwandb run: {run.url if run else "(offline)"}')
print(f'GRPO model: https://huggingface.co/{GRPO_ADAPTER_REPO}')

## What this notebook produces (artifacts list)

After a clean run, you'll have:

| Artifact | Where | Why it matters |
|---|---|---|
| **Trained LoRA adapter** | `huggingface.co/dakshdoesdev/sre-gym-qwen35-4b-grpo` (HF Hub) + `/content/drive/MyDrive/sre-gym-grpo/` (Drive) | The deliverable. Defense in depth — push fails to one survives in the other. |
| **wandb run URL** | printed at end (`run.url`) | The training curve. Reward mean per step with `baseline/*` horizontal lines for random / heuristic / Llama-70B / Claude. This is the chart that goes in the pitch. |
| **Trajectory snapshots** | `training_artifacts/snapshot_step{0,50,100,150,200,250}.jsonl` (in the model repo on HF Hub) | Full rollouts at scheduled steps. Lets you show "at step 0 the model rolled back the wrong service; at step 250 it correctly identified the decoy." That's the demo moment. |
| **Training log** | `training_artifacts/training.log` (in the model repo) | Line-by-line stdout grep target. Survives Colab disconnect. |
| **Eval comparison JSON** | `training_artifacts/eval_comparison.json` (in the model repo) | Side-by-side table of `trained_3b_grpo` vs `random` on held-out scenarios + the BASELINES dict from the README. Drop straight into the pitch slide. |

## Verification checklist

- [ ] Cell 5 (pool server) prints `{"ok": true, ...}` with 30 scenarios listed
- [ ] Cell 7 prints `HF_TOKEN set: yes` and `Drive ckpt: yes -> /content/drive/MyDrive/sre-gym-grpo` (Drive prompt may interrupt — accept it)
- [ ] Cell 9 (model load) succeeds without OOM
- [ ] Cell 15 (main loop) shows `running_reward` climbing from ~0.15 to > 0.55 over 300 steps; wandb chart shows trained curve crossing baseline lines
- [ ] At least one trajectory snapshot lands in `/content/grpo_snapshots/`
- [ ] At least one Drive checkpoint exists at `/content/drive/MyDrive/sre-gym-grpo/step*/`
- [ ] Cell 17 prints `eval comparison uploaded -> .../training_artifacts/eval_comparison.json`
- [ ] Final held-out eval `trained_3b_grpo` mean > `random` mean (sanity check that learning happened)

## Caveats

- This is a **simple policy gradient** (no PPO-clip, no reference-model KL). Good enough for a first training curve. For full GRPO-with-clip+KL, extend `grpo_step()` with a frozen reference copy of the SFT model and compute `min(ratio * adv, clip(ratio) * adv) - kl_coeff * kl`.
- **Sequential rollouts** are slow (K × episode_time per step). A more efficient implementation batches generation across K prompts at once. ~4-6h for 300 steps on A100 40GB at K=4.
- **If `running_reward` stays flat past step 100**: probable causes (in order of likelihood) — (1) all K rollouts tie at the max-tick fallback (advantages all zero), fix by raising temperature to 0.9; (2) LR too low, raise to 1e-5; (3) SFT checkpoint is overfit (action variance ~0), regenerate SFT with stronger dropout. The `score_std` and `adv_spread` charts in wandb will tell you which.